In [23]:
import pandas as pd
total_deltas = pd.read_csv('total_deltas.csv')
shap_matrix = pd.read_csv('shap_matrix.csv')
prediction_results = pd.read_csv('prediction_results_by_hadm.csv')

total_data_full = pd.read_parquet('ind_hosp.parquet')
correct_predictions = prediction_results[prediction_results['correct_prediction'] == True]
test_pairs = correct_predictions[['subject_id', 'hadm_id']].drop_duplicates()
test_pairs_tuple = list(zip(test_pairs['subject_id'], test_pairs['hadm_id']))

total_data = total_data_full[
    total_data_full[['subject_id', 'hadm_id']].apply(tuple, axis=1).isin(test_pairs_tuple)
].copy()

print(f"Data filtered to test hospitalizations only:")
print(f"  Total rows: {len(total_data)}")
print(f"  Unique patients: {total_data['subject_id'].nunique()}")
print(f"  Unique hospitalizations: {total_data['hadm_id'].nunique()}")

Data filtered to test hospitalizations only:
  Total rows: 237684
  Unique patients: 33311
  Unique hospitalizations: 58385


In [24]:
agg_dict = {}

for col in total_data.columns:
    if col in ['subject_id', 'hadm_id']:
        continue
    
    if col.startswith('icd_'):
        agg_dict[col] = 'max'
    
    elif col.startswith('ccsr_'):
        agg_dict[col] = 'max'
    
    elif col.startswith('lab_') and col.endswith('_daily'):
        agg_dict[col] = 'mean'
    
    elif col in ['cumulative_procedures', 'cumulative_medications']:
        agg_dict[col] = 'mean'
    
    elif col in ['num_medications_daily', 'num_procedures_daily']:
        agg_dict[col] = 'mean'
    
    elif col in ['gender_male', 'age', 'los', 'num_diagnoses', 'num_chronic', 
                 'comorbidity_score', 'prior_admissions_12mo', 
                 'admission_type', 'discharge_location', 'insurance', 'race',
                 'admission_location']:
        agg_dict[col] = 'first'
    
    elif col == 'target_readmission_30d':
        agg_dict[col] = 'max'
    
    else:
        agg_dict[col] = 'first'

print(f"Aggregating {len(agg_dict)} columns...")
total_data = total_data.groupby(['subject_id', 'hadm_id']).agg(agg_dict).reset_index()
total_data.head(5)

Aggregating 136 columns...


,subject_id,hadm_id,dischtime,los,age,day,current_date,num_diagnoses,num_chronic,ccsr_FAC021,...,admission_location_EMERGENCY ROOM,admission_location_INFORMATION NOT AVAILABLE,admission_location_INTERNAL TRANSFER TO OR FROM PSYCH,admission_location_PACU,admission_location_PHYSICIAN REFERRAL,admission_location_PROCEDURE SITE,admission_location_TRANSFER FROM HOSPITAL,admission_location_TRANSFER FROM SKILLED NURSING FACILITY,admission_location_WALK-IN/SELF REFERRAL,gender_male
0,10000764,27897940,2132-10-19,4,86,1,2132-10-14,76,32,1,...,False,False,False,False,False,False,True,False,False,1
1,10001492,27463908,2136-09-25,1,71,1,2136-09-23,4,4,0,...,True,False,False,False,False,False,False,False,False,0
2,10001667,22672901,2173-08-24,1,86,1,2173-08-22,13,6,1,...,False,False,False,False,False,False,True,False,False,0
3,10001725,25563031,2110-04-14,2,46,1,2110-04-11,36,18,1,...,False,False,False,True,False,False,False,False,False,0
4,10001860,21441082,2188-03-30,3,84,1,2188-03-27,36,9,1,...,True,False,False,False,False,False,False,False,False,1


In [25]:
lab_names = {
    '50983': 'Sodium',
    '50971': 'Potassium',
    '50902': 'Chloride',
    '50882': 'Bicarbonate',
    '50912': 'Creatinine',
    '51006': 'BUN',
    '50931': 'Glucose',
    '50893': 'Calcium',
    '50868': 'Anion Gap',
    '51222': 'Hemoglobin',
    '51301': 'WBC',
    '51265': 'Platelet Count',
    '51221': 'Hematocrit',
    '51250': 'MCV',
    '51277': 'RDW',
    '50960': 'Magnesium',
    '50970': 'Phosphate',
    '51248': 'MCH',
    '51249': 'MCHC',
    '51279': 'RBC'
}

icd_names = {
    'I10': 'Essential (primary) hypertension',
    'E785': 'Hyperlipidemia, unspecified',
    'K219': 'Gastroesophageal reflux disease without esophagitis',
    'Z87891': 'Personal history of nicotine dependence',
    'I2510': 'Atherosclerotic heart disease of native coronary artery without angina pectoris',
    'N179': 'Acute kidney failure, unspecified',
    'F329': 'Major depressive disorder, single episode, unspecified',
    'I4891': 'Unspecified atrial fibrillation',
    'Z7901': 'Long term (current) use of anticoagulants',
    'F419': 'Anxiety disorder, unspecified',
    'E119': 'Type 2 diabetes mellitus without complications',
    'E039': 'Hypothyroidism, unspecified',
    'Z794': 'Long term (current) use of insulin',
    'D649': 'Anemia, unspecified',
    'N390': 'Urinary tract infection, site not specified'
}

ccsr_names = {
    'FAC021': 'Personal/family history of disease',
    'FAC025': 'Other specified status',
    'END011': 'Fluid and electrolyte disorders',
    'CIR011': 'Coronary atherosclerosis and other heart disease',
    'END010': 'Disorders of lipid metabolism',
    'CIR007': 'Essential hypertension',
    'END003': 'Diabetes mellitus with complication',
    'CIR019': 'Heart failure',
    'DIG004': 'Esophageal disorders',
    'CIR017': 'Cardiac dysrhythmias',
    'CIR008': 'Hypertension with complications and secondary hypertension',
    'BLD003': 'Aplastic anemia',
    'EXT027': 'External cause codes: place of occurrence of the external cause',
    'GEN002': 'Acute and unspecified renal failure',
    'END009': 'Obesity'
}

def map_feature_name(feature_name):
    if feature_name.startswith('lab_') and feature_name.endswith('_daily'):
        code = feature_name.replace('lab_', '').replace('_daily', '')
        if code in lab_names:
            return f"{lab_names[code]} ({code})"
    elif feature_name.startswith('icd_'):
        code = feature_name.replace('icd_', '')
        if code in icd_names:
            return f"{icd_names[code]} ({code})"
    elif feature_name.startswith('ccsr_'):
        code = feature_name.replace('ccsr_', '')
        if code in ccsr_names:
            return f"{ccsr_names[code]} ({code})"
    return feature_name

In [26]:
def get_patient_analysis(subject_id, hadm_id):
    patient_delta = total_deltas[
        (total_deltas['subject_id'] == subject_id) & 
        (total_deltas['hadm_id'] == hadm_id)
    ].copy()
    print(f"Actual risk: {patient_delta['risk_actual'].iloc[0]:.4f}")
    patient_shap = shap_matrix[shap_matrix['hadm_id'] == hadm_id].copy()
    shap_features = patient_shap.drop('hadm_id', axis=1).T
    shap_features.columns = ['shap_value']
    shap_features['feature'] = shap_features.index
    
    combined = patient_delta.merge(shap_features, on='feature', how='inner')
    combined['delta_abs'] = combined['delta'].abs()
    combined['shap_abs'] = combined['shap_value'].abs()
    
    return {
        'patient_id': subject_id,
        'hadm_id': hadm_id,
        'n_days': patient_delta['n_days'].iloc[0],
        'risk_actual': patient_delta['risk_actual'].iloc[0],
        'combined': combined,
        'by_delta': combined.sort_values('delta_abs', ascending=False),
        'by_shap': combined.sort_values('shap_abs', ascending=False)
    }

def interpret_delta(delta, mode_direction):
    if mode_direction == 'add':
        if delta > 0:
            return "without → ↑ risk"
        elif delta < 0:
            return "without → ↓ risk"

    elif mode_direction == 'remove':
        if delta < 0:
            return "with → ↓ risk"
        elif delta > 0:
            return "with → ↑ risk"

    elif mode_direction == 'increase_to_mean':
        if delta < 0:
            return "lower than mean → ↓ risk"
        elif delta > 0:
            return "lower than mean → ↑ risk"
    
    elif mode_direction == 'reduce_to_mean':
        if delta < 0:
            return "higher than mean → ↓ risk"
        elif delta > 0:
            return "higher than mean → ↑ risk"
    
    elif mode_direction == 'low':
        if delta > 0:
            return "low → ↑ risk"
        elif delta < 0:
            return "low → ↓ risk"

    elif mode_direction == 'high':
        if delta > 0:
            return "high → ↑ risk"
        elif delta < 0:
            return "high → ↓ risk"
    
    elif mode_direction == 'normal':
        if delta > 0:
            return "normal range → ↑ risk"
        elif delta < 0:
            return "normal range → ↓ risk"

    elif mode_direction == 'older':
        if delta < 0:
            return "older → ↓ risk"
        elif delta > 0:
            return "older → ↑ risk"
    
    elif mode_direction == 'younger':
        if delta > 0:
            return "younger → ↑ risk"
        elif delta < 0:
            return "younger → ↓ risk"

    elif mode_direction == 'opposite':
        if delta > 0:
            return "↑ risk"
        elif delta < 0:
            return "↓ risk"

In [27]:
patient_info = total_data.copy()
icd_cols = [col for col in total_data.columns if col.startswith('icd_')]
ccsr_cols = [col for col in total_data.columns if col.startswith('ccsr_')]
lab_cols = [col for col in total_data.columns if col.startswith('lab_') and col.endswith('_daily')]

patient_info['icd_count'] = patient_info[icd_cols].sum(axis=1)
patient_info['ccsr_count'] = patient_info[ccsr_cols].sum(axis=1)
patient_info['lab_count'] = patient_info[lab_cols].notna().sum(axis=1)

patient_info = patient_info[
    (patient_info['icd_count'] >= 1) & (patient_info['ccsr_count'] >= 1) &
    (patient_info['lab_count'] >= 1)
].copy()

age_groups = {
    'young': (18, 40),
    'middle': (40, 65),
    'elderly': (65, 80),
    'very_elderly': (80, 120)
}

def get_age_group(age):
    for group, (low, high) in age_groups.items():
        if low <= age < high:
            return group
    return 'unknown'

patient_info = patient_info[['subject_id', 'hadm_id', 'gender_male', 'age', 'los', 'icd_count', 'ccsr_count', 'lab_count', 'target_readmission_30d']]
patient_info['age_group'] = patient_info['age'].apply(get_age_group)

print(f"Patient info created:")
print(f"  Total hospitalizations: {len(patient_info)}")
print(f"  With readmission: {patient_info['target_readmission_30d'].sum():.0f}")
print(f"  Without readmission: {len(patient_info) - patient_info['target_readmission_30d'].sum():.0f}")

def get_stratified_sample(patient_info, sample_size=20, random_state=42):
    readmission_yes = patient_info[patient_info['target_readmission_30d'] == 1]
    readmission_no = patient_info[patient_info['target_readmission_30d'] == 0]
    
    n_total = sample_size
    n_yes = n_total // 2
    n_no = n_total - n_yes
    samples = []

    for group, label, n in [(readmission_yes, 'with_readmission', n_yes), 
                            (readmission_no, 'without_readmission', n_no)]:
       
        age_groups_list = group['age_group'].unique()
        n_per_age = n // len(age_groups_list) if len(age_groups_list) > 0 else 0
        remainder = n - n_per_age * len(age_groups_list)
        
        for i, age_group in enumerate(age_groups_list):
            age_group_patients = group[group['age_group'] == age_group]

            n_age = n_per_age + (1 if i < remainder else 0)
            if n_age == 0:
                continue

            male = age_group_patients[age_group_patients['gender_male'] == 1]
            female = age_group_patients[age_group_patients['gender_male'] == 0]
            
            n_male_age = min(n_age // 2, len(male))
            n_female_age = n_age - n_male_age
            
            if len(male) < n_male_age:
                n_male_age = len(male)
                n_female_age = n_age - n_male_age
            
            if len(female) < n_female_age:
                n_female_age = len(female)
                n_male_age = n_age - n_female_age
            
            if len(male) > 0 and n_male_age > 0:
                sample_male = male.sample(n=min(n_male_age, len(male)), random_state=random_state)
                samples.append(sample_male)
            
            if len(female) > 0 and n_female_age > 0:
                sample_female = female.sample(n=min(n_female_age, len(female)), random_state=random_state)
                samples.append(sample_female)

    if len(samples) > 0:
        sample = pd.concat(samples).sample(frac=1, random_state=random_state)
    else:
        print("No samples selected")
        return pd.DataFrame()
    
    print(f"\nSelected {len(sample)} hospitalizations")
    print(f"   With readmission: {sample['target_readmission_30d'].sum():.0f}")
    print(f"   Without readmission: {len(sample) - sample['target_readmission_30d'].sum():.0f}")
    print(f"   Male: {sample['gender_male'].sum():.0f}")
    print(f"   Female: {len(sample) - sample['gender_male'].sum():.0f}")
    
    print(f"\nAge groups in sample:")
    print(sample['age_group'].value_counts().sort_index())
    
    return sample

SAMPLE_SIZE = 20
sample_patients_df = get_stratified_sample(patient_info, sample_size=SAMPLE_SIZE, random_state=13)
sample_patients_df

Patient info created:
  Total hospitalizations: 34892
  With readmission: 6498
  Without readmission: 28394

Selected 20 hospitalizations
   With readmission: 10
   Without readmission: 10
   Male: 8
   Female: 12

Age groups in sample:
age_group
elderly         5
middle          6
very_elderly    5
young           4
Name: count, dtype: int64


,subject_id,hadm_id,gender_male,age,los,icd_count,ccsr_count,lab_count,target_readmission_30d,age_group
11066,11897486,23193352,0,91,4,4.0,3,17,0,very_elderly
21458,13697787,22246602,0,83,6,5.0,6,20,0,very_elderly
55312,19450100,25314766,1,66,4,2.0,3,20,1,elderly
31052,15363935,24356132,0,65,2,3.0,4,11,1,elderly
37272,16393879,28139528,1,32,5,2.0,1,20,1,young
9956,11719852,29694624,0,63,9,2.0,4,20,1,middle
8550,11490720,24212550,0,89,18,5.0,8,20,1,very_elderly
25640,14413352,20235742,0,54,3,2.0,2,20,0,middle
37151,16380214,20860234,0,50,3,1.0,2,20,0,middle
42634,17306425,21245006,1,59,7,2.0,4,20,0,middle


In [28]:
def get_patient_full_data_with_analysis(subject_id, hadm_id, total_data):

    patient_raw = total_data[
        (total_data['subject_id'] == subject_id) & 
        (total_data['hadm_id'] == hadm_id)
    ]

    patient = patient_raw.iloc[0].to_dict()
    
    clean_data = {
        'subject_id': subject_id,
        'hadm_id': hadm_id,
        'age': patient.get('age'),
        'gender': 'Male' if patient.get('gender_male') == 1 else 'Female',
        'length_of_stay': patient.get('los'),
        'readmission': 'Yes' if patient.get('target_readmission_30d') == 1 else 'No',
    }

    icd_cols = [col for col in total_data.columns if col.startswith('icd_')]
    ccsr_cols = [col for col in total_data.columns if col.startswith('ccsr_')]
    lab_cols = [col for col in total_data.columns if col.startswith('lab_') and col.endswith('_daily')]
    
    all_features_original = icd_cols + ccsr_cols + lab_cols + [
        'num_diagnoses', 'num_chronic', 'comorbidity_score',
        'num_medications_daily', 'prior_admissions_12mo',
        'cumulative_procedures', 'cumulative_medications',
        'num_procedures_daily', 'gender_male', 'age'
    ]
    
    icd_present = [col for col in icd_cols if col in patient and patient[col] == 1]
    ccsr_present = [col for col in ccsr_cols if col in patient and patient[col] == 1]
    lab_present = {col: patient[col] for col in lab_cols if col in patient and pd.notna(patient[col])}
    
    clean_data['icd_diagnoses'] = [map_feature_name(col) for col in icd_present]
    clean_data['icd_count'] = len(icd_present)
    
    clean_data['ccsr_diagnoses'] = [map_feature_name(col) for col in ccsr_present]
    clean_data['ccsr_count'] = len(ccsr_present)
    
    clean_data['lab_values'] = {map_feature_name(col): val for col, val in lab_present.items()}
    clean_data['lab_count'] = len(lab_present)
    
    other_features = ['num_diagnoses', 'num_chronic', 'comorbidity_score',
                      'num_medications_daily', 'prior_admissions_12mo',
                      'cumulative_procedures', 'cumulative_medications',
                      'num_procedures_daily']
    
    for feat in other_features:
        if feat in patient and pd.notna(patient[feat]):
            clean_data[feat] = patient[feat]
    
    race_cols = [col for col in total_data.columns if col.startswith('race_')]
    for col in race_cols:
        if col in patient and patient[col] == 1:
            clean_data['race'] = col.replace('race_', '').replace('_', ' ').title()
            break
    if 'race' not in clean_data:
        clean_data['race'] = 'Unknown'
    
    insurance_cols = [col for col in total_data.columns if col.startswith('insurance_')]
    for col in insurance_cols:
        if col in patient and patient[col] == 1:
            clean_data['insurance'] = col.replace('insurance_', '')
            break
    if 'insurance' not in clean_data:
        clean_data['insurance'] = 'Unknown'
    
    admission_type_cols = [col for col in total_data.columns if col.startswith('admission_type_')]
    for col in admission_type_cols:
        if col in patient and patient[col] == 1:
            clean_data['admission_type'] = col.replace('admission_type_', '').replace('_', ' ').title()
            break
    if 'admission_type' not in clean_data:
        clean_data['admission_type'] = 'Unknown'
    
    discharge_cols = [col for col in total_data.columns if col.startswith('discharge_location_')]
    for col in discharge_cols:
        if col in patient and patient[col] == 1:
            clean_data['discharge_location'] = col.replace('discharge_location_', '').replace('_', ' ').title()
            break
    if 'discharge_location' not in clean_data:
        clean_data['discharge_location'] = 'Unknown'
    
    adm_location_cols = [col for col in total_data.columns if col.startswith('admission_location_')]
    for col in adm_location_cols:
        if col in patient and patient[col] == 1:
            clean_data['admission_location'] = col.replace('admission_location_', '').replace('_', ' ').title()
            break
    if 'admission_location' not in clean_data:
        clean_data['admission_location'] = 'Unknown'

    result = get_patient_analysis(subject_id, hadm_id)
    
    if result:
        result['combined'] = result['combined'][
            result['combined']['feature'].isin(all_features_original)
        ]
        
        result['by_delta'] = result['combined'].sort_values('delta_abs', ascending=False)
        result['by_shap'] = result['combined'].sort_values('shap_abs', ascending=False)
        
        result['combined']['feature_display'] = result['combined']['feature'].apply(map_feature_name)
        result['by_delta']['feature_display'] = result['by_delta']['feature'].apply(map_feature_name)
        result['by_shap']['feature_display'] = result['by_shap']['feature'].apply(map_feature_name)
        
        if len(result['by_delta']) > 0:
            result['by_delta']['interpretation'] = result['by_delta'].apply(
                lambda row: interpret_delta(row['delta'], row['mode_direction']), axis=1
            )
        
        if len(result['by_shap']) > 0:
            result['by_shap']['interpretation'] = result['by_shap'].apply(
                lambda row: interpret_delta(row['delta'], row['mode_direction']), axis=1
            )
        clean_data['analysis'] = result
    
    return clean_data

In [29]:
import json
import numpy as np
import pandas as pd

def convert_to_serializable(obj):
    if isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    elif isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, pd.Series):
        return obj.to_dict()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, pd.DataFrame):
        return obj.to_dict('records')
    elif pd.isna(obj):
        return None
    else:
        return obj

def extract_patient_data_only(subject_id, hadm_id, total_data):
    clean_data = get_patient_full_data_with_analysis(subject_id, hadm_id, total_data)
    if clean_data is None:
        print(f"No data found for patient {subject_id}, {hadm_id}")
        return None
    
    clean_data.pop('analysis', None)
    
    clean_data.pop('icd_count', None)
    clean_data.pop('ccsr_count', None)
    clean_data.pop('lab_count', None)
    
    patient_record = {
        'subject_id': subject_id,
        'hadm_id': hadm_id,
        'demographics': {
            'age': clean_data.get('age'),
            'gender': clean_data.get('gender'),
            'race': clean_data.get('race', 'Unknown'),
            'insurance': clean_data.get('insurance', 'Unknown'),
            'length_of_stay_days': clean_data.get('length_of_stay')
        },
        'admission': {
            'type': clean_data.get('admission_type', 'Unknown'),
            'location': clean_data.get('admission_location', 'Unknown'),
            'discharge_location': clean_data.get('discharge_location', 'Unknown')
        },
        'diagnoses': {
            'icd': clean_data.get('icd_diagnoses', []),
            'ccsr': clean_data.get('ccsr_diagnoses', [])
        },
        'laboratory_values': clean_data.get('lab_values', {}),
        'clinical_indicators': {
            'num_diagnoses': clean_data.get('num_diagnoses', 0),
            'num_chronic': clean_data.get('num_chronic', 0),
            'comorbidity_score': clean_data.get('comorbidity_score', 0),
            'num_medications_daily': clean_data.get('num_medications_daily', 0),
            'prior_admissions_12mo': clean_data.get('prior_admissions_12mo', 0),
            'cumulative_procedures': clean_data.get('cumulative_procedures', 0),
            'cumulative_medications': clean_data.get('cumulative_medications', 0),
            'num_procedures_daily': clean_data.get('num_procedures_daily', 0)
        }
    }
    
    return convert_to_serializable(patient_record)


def save_all_patients_to_single_json(sample_patients_df, total_data, output_file='all_patients.json'):
    all_patients = []
    failed_patients = []
    print(f"Processing {len(sample_patients_df)} patients...")

    for idx, row in sample_patients_df.iterrows():
        subject_id = row['subject_id']
        hadm_id = row['hadm_id']
        
        patient_data = extract_patient_data_only(subject_id, hadm_id, total_data)
        
        if patient_data:
            all_patients.append(patient_data)
            print(f"Patient {idx+1}/{len(sample_patients_df)}: {subject_id}_{hadm_id}")
        else:
            failed_patients.append(f"{subject_id}_{hadm_id}")
            print(f"Patient {idx+1}/{len(sample_patients_df)}: {subject_id}_{hadm_id} - FAILED")
    
    output_data = {
        'total_patients': len(all_patients),
        'failed_patients': failed_patients,
        'patients': all_patients
    }
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)
    
    print(f"Success: {len(all_patients)} patients saved to: {output_file}")
    if failed_patients:
        print(f"Failed: {len(failed_patients)} patients")
    
    return output_data

all_patients_data = save_all_patients_to_single_json(
    sample_patients_df,
    total_data,
    output_file='all_patients.json'
)

Processing 20 patients...
Actual risk: 0.3310
Patient 11067/20: 11897486_23193352
Actual risk: 0.7184
Patient 21459/20: 13697787_22246602
Actual risk: 0.7654
Patient 55313/20: 19450100_25314766
Actual risk: 0.5395
Patient 31053/20: 15363935_24356132
Actual risk: 0.7907
Patient 37273/20: 16393879_28139528
Actual risk: 0.9753
Patient 9957/20: 11719852_29694624
Actual risk: 0.9947
Patient 8551/20: 11490720_24212550
Actual risk: 0.4828
Patient 25641/20: 14413352_20235742
Actual risk: 0.2403
Patient 37152/20: 16380214_20860234
Actual risk: 0.5318
Patient 42635/20: 17306425_21245006
Actual risk: 0.8216
Patient 38181/20: 16536632_27119150
Actual risk: 0.0744
Patient 56441/20: 19654579_23780957
Actual risk: 0.8894
Patient 33639/20: 15794497_28987036
Actual risk: 0.7618
Patient 20261/20: 13509515_29941380
Actual risk: 0.9972
Patient 28261/20: 14872672_29294271
Actual risk: 0.9614
Patient 107/20: 10017882_20281726
Actual risk: 0.2409
Patient 14511/20: 12487653_25481295
Actual risk: 0.2389
Patien

In [30]:
import json

def extract_shap_delta_analysis(subject_id, hadm_id, total_data):
    clean_data = get_patient_full_data_with_analysis(subject_id, hadm_id, total_data)
    
    if clean_data is None:
        print(f"No data found for patient {subject_id}, {hadm_id}")
        return None
    
    analysis = clean_data.get('analysis', {})
    
    if not analysis:
        print(f"No analysis found for patient {subject_id}, {hadm_id}")
        return None
    
    shap_delta_record = {
        'subject_id': subject_id,
        'hadm_id': hadm_id,
        'risk_actual': float(analysis.get('risk_actual', 0)),
        'readmission_30d': clean_data.get('readmission'),
        'n_days': int(analysis.get('n_days', 0)),
        'shap_values': {},
        'delta_values': {},
        'delta_interpretations': {},
    }
    
    if 'by_shap' in analysis and len(analysis['by_shap']) > 0:
        for _, row in analysis['by_shap'].iterrows():
            feature = row.get('feature_display', row.get('feature'))
            shap_val = float(row.get('shap_value', 0))
            shap_delta_record['shap_values'][feature] = shap_val
    
    if 'by_delta' in analysis and len(analysis['by_delta']) > 0:
        for _, row in analysis['by_delta'].iterrows():
            feature = row.get('feature_display', row.get('feature'))
            delta_val = float(row.get('delta', 0))
            shap_delta_record['delta_values'][feature] = delta_val
            
            if 'interpretation' in row:
                shap_delta_record['delta_interpretations'][feature] = row['interpretation']
    
    return convert_to_serializable(shap_delta_record)


def save_all_shap_delta_to_json(sample_patients_df, total_data, output_file='shap_delta_analysis.json'):
    all_analysis = []
    failed_patients = []
    print(f"Processing SHAP/Delta for {len(sample_patients_df)} patients...")
    
    for idx, row in sample_patients_df.iterrows():
        subject_id = row['subject_id']
        hadm_id = row['hadm_id']
        
        analysis_data = extract_shap_delta_analysis(subject_id, hadm_id, total_data)
        
        if analysis_data:
            all_analysis.append(analysis_data)
            print(f"Patient {idx+1}/{len(sample_patients_df)}: {subject_id}_{hadm_id} "
                  f"(risk: {analysis_data['risk_actual']:.4f}, "
                  f"features: {len(analysis_data['delta_values'])})")
        else:
            failed_patients.append(f"{subject_id}_{hadm_id}")
            print(f"Patient {idx+1}/{len(sample_patients_df)}: {subject_id}_{hadm_id} - FAILED")
    
    output_data = {
        'total_patients': len(all_analysis),
        'failed_patients': failed_patients,
        'patients_analysis': all_analysis
    }
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)
    
    print(f"Success: {len(all_analysis)} patients SHAP/Delta saved to: {output_file}")
    if failed_patients:
        print(f"Failed: {len(failed_patients)} patients")
    
    return output_data

shap_delta_data = save_all_shap_delta_to_json(
    sample_patients_df,
    total_data,
    output_file='shap_delta_analysis.json'
)

Processing SHAP/Delta for 20 patients...
Actual risk: 0.3310
Patient 11067/20: 11897486_23193352 (risk: 0.3310, features: 60)
Actual risk: 0.7184
Patient 21459/20: 13697787_22246602 (risk: 0.7184, features: 60)
Actual risk: 0.7654
Patient 55313/20: 19450100_25314766 (risk: 0.7654, features: 60)
Actual risk: 0.5395
Patient 31053/20: 15363935_24356132 (risk: 0.5395, features: 60)
Actual risk: 0.7907
Patient 37273/20: 16393879_28139528 (risk: 0.7907, features: 60)
Actual risk: 0.9753
Patient 9957/20: 11719852_29694624 (risk: 0.9753, features: 60)
Actual risk: 0.9947
Patient 8551/20: 11490720_24212550 (risk: 0.9947, features: 60)
Actual risk: 0.4828
Patient 25641/20: 14413352_20235742 (risk: 0.4828, features: 60)
Actual risk: 0.2403
Patient 37152/20: 16380214_20860234 (risk: 0.2403, features: 60)
Actual risk: 0.5318
Patient 42635/20: 17306425_21245006 (risk: 0.5318, features: 60)
Actual risk: 0.8216
Patient 38181/20: 16536632_27119150 (risk: 0.8216, features: 60)
Actual risk: 0.0744
Patien